In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import random
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import gc
gc.collect()

0

In [2]:

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# drive.mount('/content/drive')

Using device: cuda


In [3]:

PAD_TOKEN = 0
START_TOKEN = 1
END_TOKEN = 2
UNK_TOKEN = 3


HIDDEN_SIZE = 100
NUM_LAYERS = 1

LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 10
EMBEDDING_DIM =200


HINDI_FASTTEXT ='/kaggle/input/datasets/kgcodex/embeddings/wiki.hi.vec'
ENGLISH_GLOVE = '/kaggle/input/datasets/kgcodex/embeddings/glove.6B.200d.txt' 
GLOVE_DIM=200
FASTEXT_DIM=300

In [4]:
def tokenize(text):
    return str(text).lower().split()
def build_vocab(sentences, min_freq=10):
    word_freq = Counter()
    for sentence in sentences:
        words = str(sentence).lower().split()
        word_freq.update(words)

    vocab = {"<pad>": PAD_TOKEN, "<start>": START_TOKEN, "<end>": END_TOKEN, "<unk>": UNK_TOKEN}
    for word, freq in word_freq.items():
        if freq > min_freq and word not in vocab:
            vocab[word] = len(vocab)
        

    return vocab
  

In [5]:

train_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_train.csv').dropna()
valid_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_valid.csv').dropna()
test_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_valid.csv').dropna()

eng_vocab = build_vocab(train_df['source'].tolist())
ind_vocab = build_vocab(train_df['target'].tolist())
reverse_ind_vocab = {idx: word for word, idx in ind_vocab.items()}

print(f"English Vocab Size: {len(eng_vocab)} | Indian Vocab Size: {len(ind_vocab)}")

English Vocab Size: 9244 | Indian Vocab Size: 8654


In [6]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, df, eng_vocab, ind_vocab):
        self.df = df
        self.eng_vocab = eng_vocab
        self.ind_vocab = ind_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
  
        src_sent = str(self.df.iloc[idx]['source']).lower().split()
        trg_sent = str(self.df.iloc[idx]['target']).lower().split()

    
        en_indices = [self.eng_vocab.get(word, 3) for word in src_sent]

       
        in_indices = [1] + [self.ind_vocab.get(word, 3) for word in trg_sent] + [2]

        return torch.tensor(en_indices), torch.tensor(in_indices)
def padding_fn(batch):
        en_list, in_list = [], []

        for en_item, in_item in batch:
            en_list.append(en_item)
            in_list.append(in_item)

       
        en_padded = pad_sequence(en_list, padding_value=0)
        in_padded = pad_sequence(in_list, padding_value=0)

        return en_padded, in_padded

In [7]:


train_dataset = TranslationDataset(train_df, eng_vocab, ind_vocab)
valid_dataset = TranslationDataset(valid_df, eng_vocab, ind_vocab)
test_dataset = TranslationDataset(test_df, eng_vocab, ind_vocab)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=padding_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, collate_fn=padding_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=padding_fn)


In [8]:
def get_emb_matrix(file_path, vocab, dim, is_fasttext=True):
    matrix = np.zeros((len(vocab), dim))

    with open(file_path, 'r', encoding='utf-8') as f:
      
        if is_fasttext:
            f.readline()

        for line in f:
            parts = line.split()
      
            if len(parts) < dim:
                continue

            word = parts[0]
            if word in vocab:
                try:
                    matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                except ValueError:
                    continue

    return torch.tensor(matrix).float()

In [9]:
en_matrix = get_emb_matrix(ENGLISH_GLOVE, eng_vocab, GLOVE_DIM,is_fasttext=False)
ind_matrix = get_emb_matrix(HINDI_FASTTEXT, ind_vocab, FASTEXT_DIM , is_fasttext=True)
#

In [10]:
class LSTM_Translator(nn.Module):
    def __init__(self, eng_vocab_size, ind_vocab_size, en_matrix, in_matrix):
        super().__init__()
     
        self.en_embed = nn.Embedding.from_pretrained(en_matrix, freeze=False)
        self.encoder_lstm = nn.LSTM(input_size=200, hidden_size=512)

    
        self.in_embed = nn.Embedding.from_pretrained(in_matrix, freeze=False)
        self.decoder_lstm = nn.LSTM(input_size=300, hidden_size=512)
        self.fc_out = nn.Linear(512, ind_vocab_size)

    def forward(self, source, target):
       
        _, (hidden, cell) = self.encoder_lstm(self.en_embed(source))


       
        output, _ = self.decoder_lstm(self.in_embed(target), (hidden, cell))

      
        prediction = self.fc_out(output)
        return prediction

In [11]:
model = LSTM_Translator(len(eng_vocab), len(ind_vocab), en_matrix, ind_matrix).to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <PAD>

In [12]:
def train_model(model, data_loader, optimizer, criterion, epochs):
    model.train()

    print(f"Starting Training for {epochs} epochs...")

    for epoch in range(epochs):
        gc.collect()
        total_loss = 0
        cnt=0
        for src, trg in data_loader:
         
            src, trg = src.to(device), trg.to(device)

            optimizer.zero_grad()

            
            output = model(src, trg[:-1])

            output_dim = output.shape[-1]
            output = output.view(-1, output_dim)
            target = trg[1:].view(-1) 

            loss = criterion(output, target)
            loss.backward()

           
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

            optimizer.step()

            total_loss += loss.item()
            if cnt%500==0 :
                print("_",cnt)
            cnt=cnt+1

      
        avg_epoch_loss = total_loss / len(data_loader)
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_epoch_loss:.4f}")

    print("Training Complete!")

In [13]:
train_model(model, train_loader, optimizer, criterion, epochs=20)

Starting Training for 20 epochs...
_ 0
_ 500
_ 1000
Epoch [1/20] | Loss: 5.1800
_ 0
_ 500
_ 1000
Epoch [2/20] | Loss: 4.2864
_ 0
_ 500
_ 1000
Epoch [3/20] | Loss: 3.9158
_ 0
_ 500
_ 1000
Epoch [4/20] | Loss: 3.6568
_ 0
_ 500
_ 1000
Epoch [5/20] | Loss: 3.4450
_ 0
_ 500
_ 1000
Epoch [6/20] | Loss: 3.2600
_ 0
_ 500
_ 1000
Epoch [7/20] | Loss: 3.0893
_ 0
_ 500
_ 1000
Epoch [8/20] | Loss: 2.9300
_ 0
_ 500
_ 1000
Epoch [9/20] | Loss: 2.7775
_ 0
_ 500
_ 1000
Epoch [10/20] | Loss: 2.6332
_ 0
_ 500
_ 1000
Epoch [11/20] | Loss: 2.4951
_ 0
_ 500
_ 1000
Epoch [12/20] | Loss: 2.3647
_ 0
_ 500
_ 1000
Epoch [13/20] | Loss: 2.2409
_ 0
_ 500
_ 1000
Epoch [14/20] | Loss: 2.1238
_ 0
_ 500
_ 1000
Epoch [15/20] | Loss: 2.0130
_ 0
_ 500
_ 1000
Epoch [16/20] | Loss: 1.9089
_ 0
_ 500
_ 1000
Epoch [17/20] | Loss: 1.8121
_ 0
_ 500
_ 1000
Epoch [18/20] | Loss: 1.7215
_ 0
_ 500
_ 1000
Epoch [19/20] | Loss: 1.6349
_ 0
_ 500
_ 1000
Epoch [20/20] | Loss: 1.5558
Training Complete!


In [14]:
def translate_batch(model, src_batch, in_vocab, max_len=100):
    model.eval()
    batch_size = src_batch.shape[1] 
    inv_in_vocab = {v: k for k, v in in_vocab.items()}

    with torch.no_grad():
       
        _, (hidden, cell) = model.encoder_lstm(model.en_embed(src_batch))

       
        current_input = torch.full((1, batch_size), 1, dtype=torch.long).to(device)
        
        all_predictions = [[] for _ in range(batch_size)] 
        finished = [False] * batch_size

        for _ in range(max_len):
            output, (hidden, cell) = model.decoder_lstm(model.in_embed(current_input), (hidden, cell))
            
            logits = model.fc_out(output).squeeze(0) 
            next_tokens = logits.argmax(1)

            for i in range(batch_size):
                if not finished[i]:
                    token_idx = next_tokens[i].item()
                    if token_idx == 2: 
                        finished[i] = True
                    else:
                        all_predictions[i].append(inv_in_vocab.get(token_idx, "<UNK>"))

            current_input = next_tokens.unsqueeze(0)
            if all(finished): 
                break

    return all_predictions

In [15]:
from nltk.translate.bleu_score import corpus_bleu

def calculate_bleu_batch(data_loader, model, in_vocab):
    targets = []
    predictions = []
    inv_in_vocab = {v: k for k, v in in_vocab.items()}

    for src, trg in data_loader:
        src = src.to(device)
        batch_preds = translate_batch(model, src, in_vocab)
        predictions.extend(batch_preds)

        trg_batch = trg.cpu().numpy().T 
        for sentence_indices in trg_batch:
            clean_ref = [inv_in_vocab.get(idx, "<UNK>") for idx in sentence_indices 
                         if idx not in [0, 1, 2]]
            targets.append([clean_ref])

   
    b1 = corpus_bleu(targets, predictions, weights=(1, 0, 0, 0))
    b2 = corpus_bleu(targets, predictions, weights=(0.5, 0.5, 0, 0))
    b3 = corpus_bleu(targets, predictions, weights=(0.33, 0.33, 0.33, 0))
    b4 = corpus_bleu(targets, predictions, weights=(0.25, 0.25, 0.25, 0.25))

    print(f"BLEU-1: {b1*100:.2f} | BLEU-2: {b2*100:.2f} | BLEU-3: {b3*100:.2f} | BLEU-4: {b4*100:.2f}")

In [16]:

calculate_bleu_batch(test_loader, model, ind_vocab)

BLEU-1: 27.76 | BLEU-2: 14.14 | BLEU-3: 7.82 | BLEU-4: 4.40


In [17]:
def generate_report_examples(model, data_loader, eng_vocab, ind_vocab, num_examples=5):
    model.eval()
    inv_eng_vocab = {v: k for k, v in eng_vocab.items()}
    inv_ind_vocab = {v: k for k, v in ind_vocab.items()}
    
    examples_found = 0
    report_data = []

    with torch.no_grad():
        for src, trg in data_loader:
            if examples_found >= num_examples:
                break
            
            src = src.to(device)
           
            batch_preds = translate_batch(model, src, ind_vocab)
            
        
            src_batch = src.cpu().numpy().T
            trg_batch = trg.cpu().numpy().T
            
            for i in range(len(batch_preds)):
                if examples_found >= num_examples:
                    break
                
              
                source_sent = [inv_eng_vocab.get(idx, "<UNK>") for idx in src_batch[i] if idx > 0]
                
              
                ground_truth = [inv_ind_vocab.get(idx, "<UNK>") for idx in trg_batch[i] if idx not in [0, 1, 2]]
      
                prediction = batch_preds[i]
                
                report_data.append({
                    "Example": examples_found + 1,
                    "Source": " ".join(source_sent),
                    "Ground Truth": " ".join(ground_truth),
                    "Prediction": " ".join(prediction)
                })
                
                examples_found += 1
                
    return report_data

def print_report(title, data):
    print(f"\n{'='*20} {title} {'='*20}")
    for item in data:
        print(f"Example {item['Example']}:")
        print(f"  Source:       {item['Source']}")
        print(f"  Ground Truth: {item['Ground Truth']}")
        print(f"  Model Output: {item['Prediction']}")
        print("-" * 50)

In [18]:

train_examples = generate_report_examples(model, train_loader, eng_vocab, ind_vocab, num_examples=5)
print_report("TRAINING EXAMPLES", train_examples)


test_examples = generate_report_examples(model, test_loader, eng_vocab, ind_vocab, num_examples=5)
print_report("TEST EXAMPLES", test_examples)


==================== TRAINING EXAMPLES ====================
Example 1:
  Source:       regional directors conference of staff selection commission begins today
  Ground Truth: कर्मचारी चयन आयोग के क्षेत्रीय निदेशकों के सम्‍मेलन का आज शुभारंभ
  Model Output: मुख्य सलाहकार समिति की अध्यक्षता में कैबिनेट की बैठक आज <unk> में की गई
--------------------------------------------------
Example 2:
  Source:       such were the <unk> we relate some <unk> of them unto thee <unk> their messengers verily came unto them with clear <unk> (of allah 's <unk> but they could not believe because they had before <unk> thus doth allah print upon the hearts of disbelievers <unk> they hear <unk>
  Ground Truth: (ऐ रसूल) ये चन्द <unk> हैं जिन के हालात हम तुमसे बयान करते हैं और इसमें तो शक़ ही नहीं कि उनके पैग़म्बर उनके पास <unk> व रौशन <unk> लेकर आए मगर ये लोग जिसके पहले <unk> चुके थे उस पर भला <unk> को ईमान लाने वाले थे ख़ुदा यूं <unk> के दिलों पर <unk> <unk> कर देता है (कि ये ईमान न <unk>
  Model Output: (ऐ